# 15_milvus: interactive lesson

Read README.md first. Predict each outcome before executing. Code below is mirrored from the scripts; edit a copy to experiment. No model downloads or containers are started.

In [ ]:
import os
import sys

from common.settings import ROOT

os.chdir(ROOT)
sys.argv = ["notebook"]
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["ANONYMIZED_TELEMETRY"] = "False"

## 01_from_scratch

Source: `modules/15_milvus/examples/01_from_scratch.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
"""Distinguish an acknowledged write from its visibility to a reader.

Run: uv run python modules/15_milvus/examples/01_from_scratch.py
Concepts: Distinguish an acknowledged write from its visibility to a reader.
Expected: a stale sale and a later retirement.
CPU: about 2–90 seconds, depending on the profile and index. Models: included vectors; optional local caption model only for the generation example. Marlowe local inputs; no database server required.
"""


def visible(events, read_at, applied):
    state = {}
    for version, key, value in events:
        if version <= min(read_at, applied):
            state[key] = value
    return state


def main() -> None:
    # KEY IDEA: Distinguish an acknowledged write from its visibility to a reader.
    writes = [(1, "boot", "in stock"), (2, "boot", "retired")]
    stale = visible(writes, 2, 1)
    caught_up = visible(writes, 2, 2)
    assert stale["boot"] == "in stock" and caught_up["boot"] == "retired"
    print("reader applied through version 1", stale, "version 2", caught_up)


if __name__ == "__main__":
    main()

## 02_production

Source: `modules/15_milvus/examples/02_production.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
import importlib.util

if importlib.util.find_spec("pymilvus") is None or "milvus" not in os.environ.get(
    "MARLOWE_NOTEBOOK_PROFILES", ""
).split(","):
    print(
        "SKIP native comparison: --extra milvus; start milvus and include it in MARLOWE_NOTEBOOK_PROFILES"
    )
else:
    """Create an indexed Milvus collection and search named fields.

    Run: uv run --extra milvus python modules/15_milvus/examples/02_production.py
    Concepts: Create an indexed Milvus collection and search named fields.
    Expected: IVF collection and native result.
    CPU: about 2–90 seconds, depending on the profile and index. Models: included vectors; optional local caption model only for the generation example. Marlowe local inputs; start the milvus profile first.
    """

    # course-extra: milvus
    # course-profile: milvus
    from common.tier3 import session

    def main() -> None:
        # KEY IDEA: Create an indexed Milvus collection and search named fields.
        with session("milvus", 160, index="IVF_FLAT") as (s, _, q):
            hits = s.db.search(
                collection_name=s.name,
                data=[q.tolist()],
                anns_field="v",
                limit=5,
                search_params={"metric_type": "COSINE", "params": {"nprobe": 8}},
                output_fields=["body"],
            )
            print(
                "schema",
                s.db.describe_collection(s.name)["collection_name"],
                "indexes",
                s.db.list_indexes(s.name),
            )
            print("first hit", hits[0][0])
            assert len(hits[0]) == 5

    if __name__ == "__main__":
        from common.lesson_runtime import run_server_lesson

        run_server_lesson(main, "milvus")

## 03_measure

Source: `modules/15_milvus/examples/03_measure.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
import importlib.util

if importlib.util.find_spec("pymilvus") is None or "milvus" not in os.environ.get(
    "MARLOWE_NOTEBOOK_PROFILES", ""
).split(","):
    print(
        "SKIP native comparison: --extra milvus; start milvus and include it in MARLOWE_NOTEBOOK_PROFILES"
    )
else:
    """Measure a controlled database comparison on the same catalog.

    Run: uv run --extra milvus python modules/15_milvus/examples/03_measure.py
    Concepts: Measure a controlled database comparison on the same catalog.
    Expected: three index families.
    CPU: about 2–90 seconds, depending on the profile and index. Models: included vectors; optional local caption model only for the generation example. Marlowe local inputs; start the milvus profile first.
    """

    # course-extra: milvus
    # course-profile: milvus
    from common.tier3 import measure

    def main() -> None:
        # KEY IDEA: Measure a controlled database comparison on the same catalog.
        for kind in ("FLAT", "HNSW", "IVF_FLAT"):
            measure("milvus", parameters={"index": kind})

    if __name__ == "__main__":
        from common.lesson_runtime import run_server_lesson

        run_server_lesson(main, "milvus")

## 04_hybrid_bm25

Source: `modules/15_milvus/examples/04_hybrid_bm25.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
import importlib.util

if importlib.util.find_spec("pymilvus") is None or "milvus" not in os.environ.get(
    "MARLOWE_NOTEBOOK_PROFILES", ""
).split(","):
    print(
        "SKIP native comparison: --extra milvus; start milvus and include it in MARLOWE_NOTEBOOK_PROFILES"
    )
else:
    """Fuse a Milvus BM25 function and dense vector search.

    Run: uv run --extra milvus python modules/15_milvus/examples/04_hybrid_bm25.py
    Concepts: Fuse a Milvus BM25 function and dense vector search.
    Expected: BM25 and dense RRF.
    CPU: about 2–90 seconds, depending on the profile and index. Models: included vectors; optional local caption model only for the generation example. Marlowe local inputs; start the milvus profile first.
    """

    # course-extra: milvus
    # course-profile: milvus
    from common.tier3 import session

    def main() -> None:
        # KEY IDEA: Fuse a Milvus BM25 function and dense vector search.
        with session("milvus", 160) as (s, rows, q):
            text = rows.iloc[0].sku
            bm25 = s.db.search(
                s.name,
                data=[text],
                anns_field="terms",
                limit=5,
                search_params={"metric_type": "BM25", "params": {}},
            )
            fused = s.hybrid_search(text, q, 5)
            assert bm25[0] and fused
            print(
                "BM25", [(r["key"], r["distance"]) for r in bm25[0]], "RRF", [h.id for h in fused]
            )

    if __name__ == "__main__":
        from common.lesson_runtime import run_server_lesson

        run_server_lesson(main, "milvus")

## 05_partitions_consistency

Source: `modules/15_milvus/examples/05_partitions_consistency.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
import importlib.util

if importlib.util.find_spec("pymilvus") is None or "milvus" not in os.environ.get(
    "MARLOWE_NOTEBOOK_PROFILES", ""
).split(","):
    print(
        "SKIP native comparison: --extra milvus; start milvus and include it in MARLOWE_NOTEBOOK_PROFILES"
    )
else:
    """Measure read-after-write visibility under three Milvus policies.

    Run: uv run --extra milvus python modules/15_milvus/examples/05_partitions_consistency.py
    Concepts: Measure read-after-write visibility under three Milvus policies.
    Expected: partition and consistency observations.
    CPU: about 2–90 seconds, depending on the profile and index. Models: included vectors; optional local caption model only for the generation example. Marlowe local inputs; start the milvus profile first.
    """

    # course-extra: milvus
    # course-profile: milvus
    import time

    from pymilvus import DataType

    from common.stores.milvus import MilvusStore
    from common.teaching import evidence

    def main() -> None:
        # KEY IDEA: Measure read-after-write visibility under three Milvus policies.
        adapter = MilvusStore()
        db = adapter.db
        name = adapter.name
        schema = db.create_schema(auto_id=False)
        schema.add_field("id", DataType.INT64, is_primary=True)
        schema.add_field("tenant", DataType.VARCHAR, max_length=40, is_partition_key=True)
        schema.add_field("v", DataType.FLOAT_VECTOR, dim=2)
        idx = db.prepare_index_params()
        idx.add_index(field_name="v", index_type="FLAT", metric_type="COSINE")
        db.create_collection(name, schema=schema, index_params=idx, num_partitions=4)
        try:
            db.insert(
                name,
                [
                    {"id": 1, "tenant": "north", "v": [1.0, 0.0]},
                    {"id": 2, "tenant": "south", "v": [0.0, 1.0]},
                ],
            )
            seen = {}
            for policy in ["Eventually", "Session", "Strong"]:
                start = time.perf_counter()
                rows = db.query(
                    name,
                    filter='tenant == "north"',
                    output_fields=["id", "tenant"],
                    consistency_level=policy,
                )
                seen[policy] = {
                    "ids": [r["id"] for r in rows],
                    "milliseconds": 1000 * (time.perf_counter() - start),
                }
            assert seen["Strong"]["ids"] == [1]
            print(seen, db.list_partitions(name))
            evidence("milvus-consistency", {"observations": seen}, tier=3)
        finally:
            db.drop_collection(name)
            db.close()

    if __name__ == "__main__":
        from common.lesson_runtime import run_server_lesson

        run_server_lesson(main, "milvus")

## 06_disk_index

Source: `modules/15_milvus/examples/06_disk_index.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
import importlib.util

if importlib.util.find_spec("pymilvus") is None or "milvus" not in os.environ.get(
    "MARLOWE_NOTEBOOK_PROFILES", ""
).split(","):
    print(
        "SKIP native comparison: --extra milvus; start milvus and include it in MARLOWE_NOTEBOOK_PROFILES"
    )
else:
    """Flush real catalog vectors and inspect native DISKANN build state.

    Run: uv run --extra milvus python modules/15_milvus/examples/06_disk_index.py
    Concepts: Flush real catalog vectors and inspect native DISKANN build state.
    Expected: 4,096 indexed disk rows.
    CPU: about 2–90 seconds, depending on the profile and index. Models: included vectors; optional local caption model only for the generation example. Marlowe local inputs; start the milvus profile first.
    """

    # course-extra: milvus
    # course-profile: milvus
    import time

    from common.teaching import evidence
    from common.tier3 import session

    def main() -> None:
        # KEY IDEA: Flush real catalog vectors and inspect native DISKANN build state.
        with session("milvus", 4096, index="DISKANN") as (s, _, q):
            name = next(
                n
                for n in s.db.list_indexes(s.name)
                if s.db.describe_index(s.name, n)["field_name"] == "v"
            )
            deadline = time.monotonic() + 150
            while True:
                info = dict(s.db.describe_index(s.name, name))
                if info.get("indexed_rows", 0) >= 4096:
                    break
                if time.monotonic() > deadline:
                    raise TimeoutError(f"DISKANN did not finish indexing 4096 sealed rows: {info}")
                time.sleep(1)
            s.db.refresh_load(s.name)
            hits = s.search(q, 5)
            assert info["index_type"] == "DISKANN" and hits
            print("server index statistics", info, "top IDs", [h.id for h in hits])
            evidence("milvus-diskann", {"status": info, "ids": [h.id for h in hits]}, tier=3)

    if __name__ == "__main__":
        from common.lesson_runtime import run_server_lesson

        run_server_lesson(main, "milvus")

## Change one thing

Explain the result before and after changing the input. The code below is deliberately small enough to inspect.

In [ ]:
from common.teaching import catalog

rows, _, _, _ = catalog(100)
print(rows.groupby("region").price.median())
# Change the group or price threshold, then repeat the native comparison for 15_milvus.

## Finish

Complete Practice, Apply, Stretch, and the six-question quiz in README.md. No notebook output is committed. Preview `scripts/cleanup.py` before deleting generated course files.